In [ ]:
from pathlib import Path

# 1. Where is this notebook standing right now?
here = Path.cwd()
print("standing in :", here)

# 2. Where do we think the repo root is?
root = here if (here / "src").exists() else here.parent
print("repo root   :", root)

# 3. Does the src folder exist there, and what is inside it?
src = root / "src"
print("src exists  :", src.exists())
if src.exists():
    for f in sorted(src.iterdir()):
        print("   ", f.name)

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# 1. Find where this notebook is standing right now
here = Path.cwd()

# 2. Walk upwards, folder by folder, until we find the one that contains src
root = next((p for p in [here, *here.parents] if (p / "src").exists()), None)
if root is None:
    raise FileNotFoundError(f"No src folder found above {here}")
print("repo root:", root)

# 3. Tell Python where our scripts live, then import the loader
sys.path.append(str(root / "src"))
import load_dispatch_down as m

# 4. Load every dispatch-down file into one table (wind only, by the config)
df = m.load_all_dd()

# 5. For each group, find the first half-hour, the last, and the rows we hold
cover = df.groupby(["UT_TYPE", "JURISDICTION"])["utc"].agg(
    first="min", last="max", rows="size"
)

# 6. Work out how many half-hours SHOULD exist between first and last
cover["expected_rows"] = (
    (cover["last"] - cover["first"]) / pd.Timedelta(minutes=30)
).astype(int) + 1

# 7. Compare: expected minus actual
cover["gap"] = cover["expected_rows"] - cover["rows"]
print("total rows:", len(df))
cover

In [ ]:


# 1. Find the repo root and import our loader (same walk-upwards trick as before)
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "src").exists()), None)
sys.path.append(str(root / "src"))
import load_dispatch_down as l
import config

# 2. Load the wind data
df = l.load_all_dd()

# 3. The same three packing-slip rules the inspector uses
rules = {
    "DD_MWH": ["CONSTRAINTS_MWH", "CURTAILMENTS_MWH"],
    "CONSTRAINTS_MWH": ["TRANS_CONSTR_MWH", "TSO_TEST_MWH"],
    "CURTAILMENTS_MWH": ["HI_FRQ_MIN_GEN_MWH", "ROCOF_INERTIA_MWH", "SNSP_MWH"],
}

# 4. For each rule, store how far the total is from the sum of its parts
for total, parts in rules.items():
    df["diff_" + total] = (df[total] - df[parts].sum(axis=1)).abs()
diff_cols = [c for c in df.columns if c.startswith("diff_")]

# 5. Which rule breaks, how often, and how big is the worst difference?
summary = pd.DataFrame({
    "rows_over_tolerance": (df[diff_cols] > config.SUM_TOLERANCE).sum(),
    "largest_difference": df[diff_cols].max(),
})
print(summary)

# 6. Where do the bad rows sit? Count them per source file and jurisdiction
bad = df[(df[diff_cols] > config.SUM_TOLERANCE).any(axis=1)].copy()
print(bad.groupby(["source_file", "JURISDICTION"]).size())

# 7. Show the 5 worst rows so we can read them with our own eyes
bad["worst"] = bad[diff_cols].max(axis=1)
bad.nlargest(5, "worst")[["source_file", "JURISDICTION", "HH_TIMESTAMP",
                          "DD_MWH", "CONSTRAINTS_MWH", "CURTAILMENTS_MWH", "worst"]]

In [ ]:


# 1. Unload every dispatch-down lorry (wind only)
df = l.load_all_dd()
df["year"] = df["HH_TIMESTAMP"].dt.year

# 2. How far is output above availability? (negative means output is below availability, which is normal)
df["excess"] = df["AO_MWH"] - df["AV_MWH"]

# 3. Flag rows where output is above availability by more than the rounding allowance (0.1 MWh)
df["above"] = df["excess"] > config.SUM_TOLERANCE

# 4. Per jurisdiction and year: rows, flagged rows, and the largest excess
summary = df.groupby(["JURISDICTION", "year"]).agg(
    rows=("above", "size"),
    output_above_avail=("above", "sum"),
    largest_excess=("excess", "max"),
).round(2)
print(summary)